In [10]:
import numpy as np
import random
import matplotlib.pyplot as plt


from qiskit import transpile
from qiskit.circuit import Parameter,ParameterExpression
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit.circuit.library import QAOAAnsatz
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime.fake_provider import FakeMumbaiV2
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit_optimization.applications import Knapsack
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.circuit.library import QAOAAnsatz
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

import sys
sys.path.append("../")
from clapton.clapton import claptonize
from clapton.circuit_manipulation import transform_to_allowed_gates,qiskit_to_stim, modify_circuit, multi_angle_qaoa_circuit, generate_qiskit_param_map,relax_qaoa_parameters
from testing_scripts.qaoa_utils import evaluate_energy, QAOASolver
from testing_scripts.knapsack_utils import generate_knapsack_instance

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [11]:
# prob = Knapsack(values=[3, 4, 5, 6, 7,8,9,10,11,12], weights=[2, 3, 4, 5, 6,7,8,9,10,11], max_weight=10)

# Example usage
prob = generate_knapsack_instance(num_items=9,seed=1)

In [12]:
# prob = Knapsack(values=[3, 4, 5, 6, 7,8,9,10,11,12], weights=[2, 3, 4, 5, 6,7,8,9,10,11], max_weight=10)
qp = prob.to_quadratic_program()
print(qp.prettyprint())

Problem name: Knapsack

Maximize
  5*x_0 + 19*x_1 + 3*x_2 + 9*x_3 + 4*x_4 + 16*x_5 + 15*x_6 + 16*x_7 + 13*x_8

Subject to
  Linear constraints (1)
    13*x_0 + 4*x_1 + 2*x_2 + 8*x_3 + x_4 + 15*x_5 + 14*x_6 + 7*x_7 + 7*x_8
    <= 35  'c0'

  Binary variables (9)
    x_0 x_1 x_2 x_3 x_4 x_5 x_6 x_7 x_8



In [13]:
# intermediate QUBO form of the optimization problem
conv = QuadraticProgramToQubo()
qubo = conv.convert(qp)
print(qubo.prettyprint())

Problem name: Knapsack

Minimize
  101*c0@int_slack@0^2 + 404*c0@int_slack@0*c0@int_slack@1
  + 808*c0@int_slack@0*c0@int_slack@2 + 1616*c0@int_slack@0*c0@int_slack@3
  + 3232*c0@int_slack@0*c0@int_slack@4 + 808*c0@int_slack@0*c0@int_slack@5
  + 404*c0@int_slack@1^2 + 1616*c0@int_slack@1*c0@int_slack@2
  + 3232*c0@int_slack@1*c0@int_slack@3 + 6464*c0@int_slack@1*c0@int_slack@4
  + 1616*c0@int_slack@1*c0@int_slack@5 + 1616*c0@int_slack@2^2
  + 6464*c0@int_slack@2*c0@int_slack@3 + 12928*c0@int_slack@2*c0@int_slack@4
  + 3232*c0@int_slack@2*c0@int_slack@5 + 6464*c0@int_slack@3^2
  + 25856*c0@int_slack@3*c0@int_slack@4 + 6464*c0@int_slack@3*c0@int_slack@5
  + 25856*c0@int_slack@4^2 + 12928*c0@int_slack@4*c0@int_slack@5
  + 1616*c0@int_slack@5^2 + 2626*x_0*c0@int_slack@0 + 5252*x_0*c0@int_slack@1
  + 10504*x_0*c0@int_slack@2 + 21008*x_0*c0@int_slack@3
  + 42016*x_0*c0@int_slack@4 + 10504*x_0*c0@int_slack@5 + 17069*x_0^2
  + 10504*x_0*x_1 + 5252*x_0*x_2 + 21008*x_0*x_3 + 2626*x_0*x_4 + 39390

In [14]:
# qubit Hamiltonian and offset
op, offset = qubo.to_ising()
print(f"num qubits: {op.num_qubits}, offset: {offset}\n")
print(op)
cost_hamiltonian = op

num qubits: 15, offset: 61206.5

SparsePauliOp(['IIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIZI', 'IIIIIIIIIIIIZII', 'IIIIIIIIIIIZIII', 'IIIIIIIIIIZIIII', 'IIIIIIIIIZIIIII', 'IIIIIIIIZIIIIII', 'IIIIIIIZIIIIIII', 'IIIIIIZIIIIIIII', 'IIIIIZIIIIIIIII', 'IIIIZIIIIIIIIII', 'IIIZIIIIIIIIIII', 'IIZIIIIIIIIIIII', 'IZIIIIIIIIIIIII', 'ZIIIIIIIIIIIIII', 'IIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIZIZ', 'IIIIIIIIIIIZIIZ', 'IIIIIIIIIIZIIIZ', 'IIIIIIIIIZIIIIZ', 'IIIIIIIIZIIIIIZ', 'IIIIIIIZIIIIIIZ', 'IIIIIIZIIIIIIIZ', 'IIIIIZIIIIIIIIZ', 'IIIIZIIIIIIIIIZ', 'IIIZIIIIIIIIIIZ', 'IIZIIIIIIIIIIIZ', 'IZIIIIIIIIIIIIZ', 'ZIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIZZI', 'IIIIIIIIIIIZIZI', 'IIIIIIIIIIZIIZI', 'IIIIIIIIIZIIIZI', 'IIIIIIIIZIIIIZI', 'IIIIIIIZIIIIIZI', 'IIIIIIZIIIIIIZI', 'IIIIIZIIIIIIIZI', 'IIIIZIIIIIIIIZI', 'IIIZIIIIIIIIIZI', 'IIZIIIIIIIIIIZI', 'IZIIIIIIIIIIIZI', 'ZIIIIIIIIIIIIZI', 'IIIIIIIIIIIZZII', 'IIIIIIIIIIZIZII', 'IIIIIIIIIZIIZII', 'IIIIIIIIZIIIZII', 'IIIIIIIZIIIIZII', 'IIIIIIZIIIIIZII', 'IIIIIZIIIIIIZII', 'IIIIZIIIIIIIZII', 'I

In [ ]:
reps=2
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=reps)

In [ ]:
knapsack_qaoa = QAOASolver(cost_hamiltonian,circuit)
knapsack_qaoa.prepare_circuit()

In [ ]:
# Run CAFQA Process
knapsack_qaoa.run_CAFQA(n_gens=10)

In [ ]:
knapsack_qaoa.energy_best

In [ ]:
knapsack_qaoa.evaluate_exact_energy()

In [ ]:
cafqa_angles = [param * np.pi/2 for param in knapsack_qaoa.ks_best]

In [ ]:
energies = [evaluate_energy(knapsack_qaoa.pcirc, cost_hamiltonian, cafqa_angles) for _ in range(10)]
average_energy = np.mean(energies)
print(f"Average CAFQA Qiskit Energy: {average_energy}")

In [ ]:
# Random Initalization 
random_angles = np.random.random(len(knapsack_qaoa.ks_best))
random_energies = [evaluate_energy(knapsack_qaoa.pcirc, cost_hamiltonian, random_angles) for _ in range(10)]
min_energy = min(random_energies)
print(f"Minimum Energy found with Random initialization over 100 runs: {min_energy}")

In [ ]:
cafqa_result,cafqa_iteration_vals = knapsack_qaoa.run_qaoa(cafqa_angles,max_iters=10)
random_result,random_iteration_vals = knapsack_qaoa.run_qaoa(random_angles,max_iters=10)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(cafqa_iteration_vals, label="CAFQA")
plt.plot(random_iteration_vals, label="Random Initialization")
plt.legend()
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

# Vanilla

In [ ]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=reps)
vanilla_knapsack = QAOASolver(cost_hamiltonian,circuit.decompose().decompose())
vanilla_knapsack.vanilla = True

In [ ]:
import os
init_random_params = np.random.random(circuit.num_parameters)
res,obj_values=vanilla_knapsack.run_qaoa(init_random_params, max_iters=1000)
results_dict = {
    "vanilla_fin_energy": res,
    "vanilla_iteration_values": obj_values,
}

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(obj_values, label="Objective Values")
plt.xlabel("Iteration")
plt.ylabel("Objective Value")
plt.title("Objective Values Over Iterations")
plt.legend()
plt.show()